# aQGC constraint checker - 1.0.0

From the extracted package directory, install with `python -m pip install ".[notebook]"`, then run this notebook from the top. See `README.md` for usage and the scope of the implemented bounds.

## 1. Load the package

In [1]:
from aqgc import check, print_report

## 2. Set the inputs

Enter dimensionless Wilson coefficients with case-sensitive string keys. Both Lambda and Ecm are in TeV. The exact accepted names are:

| Sector | Accepted names |
|---|---|
| S | `FS0`, `FS1`, `FS2` |
| CP-even M | `FM0`, `FM1`, `FM2`, `FM3`, `FM4`, `FM5`, `FM7`, `FM8`, `FM9` |
| CP-even T | `FT0`, `FT1`, `FT2`, `FT3`, `FT4`, `FT5`, `FT6`, `FT7`, `FT8`, `FT9` |
| CP-odd M | `FM1odd`, `FM2odd`, `FM3odd`, `FM4odd`, `FM5odd`, `FM6odd` |
| CP-odd T | `FT1odd`, `FT2odd`, `FT3odd`, `FT4odd`, `FT5odd`, `FT6odd` |

**You do not need to enter all 34 coefficients. Every coefficient you omit is set to zero.** The example below sets FS0, FM1, FT9, and FT1odd; the remaining 30 coefficients are zero. Separate entries with commas, using `"name": value` inside `{...}`. There is no `FM6`; unknown names are rejected.

| Mode | Check |
|---|---|
| `NaiveSingle` | Single-operator unitarity |
| `NaiveCombined` | Combined-operator unitarity |
| `FinalCombined` | Positivity-constrained unitarity, including positivity |
| `Positivity` | Positivity only |

Select one mode, a list of modes, or `"All"`. Positivity alone does not require energies. All three unitarity modes contain limits for all 34 coefficients.

The Phi4 sector checks `FS0 >= 0`, `FS0 + FS2 >= 0`, and `FS0 + FS1 + FS2 >= 0` whenever positivity is evaluated. For the S operators, combined unitarity includes J=0 and J=1. The constants for FS0, FS1, FS2 are `(224/5)*pi`, `(288/5)*pi`, `(288/5)*pi` in `NaiveCombined`, and `(224/5)*pi`, `(288/5)*pi`, `48*pi` in `FinalCombined`, before the common `(Lambda / Ecm)^4` factor.

The authors independently calculated these bounds. The S-sector J=0 results agree with Physical Review D 101, 113003 (2020), arXiv:2004.05174; their S-sector positivity combinations agree with JHEP 12 (2019) 032, arXiv:1908.09845. See README.md for references, the additional J=1 tightening, and the boundary convention.


In [7]:
coefficients = {
    "FS0": 0.5,
    "FM1": 1.0,
    "FT9": 0.5,
    "FT1odd": 0.1,
}
modes = ["NaiveSingle", "NaiveCombined", "Positivity"]
lambda_tev = 1.0
ecm_tev = 2.0

## 3. Run the checks

The report shows nonzero coefficients and all non-passing conditions. The `result` dictionary retains all records.

In [10]:
result = check(
    coefficients, modes,
    lambda_tev=lambda_tev, ecm_tev=ecm_tev,
)
print_report(result)

aQGC: Pass for the selected implemented checks
Dimensionless f_i; unspecified coefficients = 0.
Lambda = 1.000000 TeV; Ecm = 2.000000 TeV

Single-operator unitarity: Pass (Pass: 34)
  FS0      f=   0.5000000  max|f|=    6.283185  Pass
  FM1      f=    1.000000  max|f|=    10.26040  Pass
  FT9      f=   0.5000000  max|f|=   0.6731984  Pass
  FT1odd   f=   0.1000000  max|f|=   0.4712389  Pass

Combined-operator unitarity: Pass (Pass: 34)
  FS0      f=   0.5000000  max|f|=    8.796459  Pass
  FM1      f=    1.000000  max|f|=    38.46864  Pass
  FT9      f=   0.5000000  max|f|=    1.570796  Pass
  FT1odd   f=   0.1000000  max|f|=   0.4712389  Pass

Positivity: Pass (Pass: 21)

Notes:
  Single-operator unitarity is an isolated-coefficient reference comparison for this multi-coefficient input.
  Combined limits are marginalized coefficient bounds, not a complete joint eigenvalue test.
  Ecm exceeds the supplied cutoff Lambda; passing these inequalities does not establish EFT validity.


## Optional output

Use `print_report(result, show_all=True)` to show every condition. To save the full result:

```python
from aqgc import export_json
export_json(result, "my_aqgc_result.json")
```

Named formulas remain available in `aqgc.definitions`, for example `unitarityNaiveSingleFM1`, `positivityB4`, and `positivityPhi4`. Unitarity names give the constant before the energy factor; positivity names contain symbolic residuals.

Inspect the S-operator positivity results with `result["positivity"]["Phi4"]`.